# 10 — The Curve (Zone Position in HTF Range)

## Goal
Score **where inside the higher-timeframe (HTF) range** a zone sits.  
Demand zones at the bottom of the range have the most room to run.  
Supply zones at the top of the range have the most room to drop.

---

## HTF Range Thirds

$$\text{third} = \frac{\text{htf\_high} - \text{htf\_low}}{3}$$

| Zone position | Demand score | Supply score |
|---------------|:---:|:---:|
| Low third (`htf_low` → `htf_low + third`) | 2 | 0 |
| Mid third | 1 | 1 |
| High third (`htf_high - third` → `htf_high`) | 0 | 2 |

"With the curve" = demand in Low, supply in High.  
"Against the curve" = demand in High, supply in Low.

---

## Zone position

The zone is placed by its **proximal** level (the edge price will touch first):

$$\text{pos} = \frac{\text{proximal} - \text{htf\_low}}{\text{htf\_high} - \text{htf\_low}} \in [0, 1]$$

$\text{pos} < 0.333$ → Low third, $\text{pos} > 0.667$ → High third, else Mid.

---

## Visualization
Horizontal bands mark the three thirds of the HTF range. Zones are coloured by curve score.


## 1. Load data and run the detection pipeline

In [ ]:
import pandas as pd
import numpy as np

# ── constants ────────────────────────────────────────────────────────────────
BASE_BODY_RATIO_MAX  = 0.50
BASE_MIN_CANDLES     = 1
BASE_MAX_CANDLES     = 5
BASE_MAX_ATR_WIDTH   = 2.5
BASE_COMPACTNESS_MAX = 0.80
LEG_CANDLES          = 3
LEG_ATR_MIN          = 1.5
DEPARTURE_CANDLES    = 3
DEPARTURE_ATR_MIN    = 1.5
DEPARTURE_RATIO_MIN  = 1.0

FORMATION_MAP = {
    ("up",   "up"):   ("RBR", "demand"),
    ("down", "down"): ("DBD", "supply"),
    ("down", "up"):   ("DBR", "demand"),
    ("up",   "down"): ("RBD", "supply"),
}

df      = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
labeled = pd.read_csv("../fixtures_labeled.csv",  index_col=0, parse_dates=True)
o_arr = df["open"].values; h_arr = df["high"].values
l_arr = df["low"].values;  c_arr = df["close"].values; atr_arr = df["atr"].values

def find_base_clusters():
    clusters, i = [], 0
    while i < len(df):
        if not df["is_base"].iloc[i]: i += 1; continue
        j = i
        while j + 1 < len(df) and df["is_base"].iloc[j + 1]: j += 1
        if BASE_MIN_CANDLES <= (j - i + 1) <= BASE_MAX_CANDLES: clusters.append((i, j))
        i = j + 1
    return clusters

def cluster_ok(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    width   = h_arr[bs:be+1].max() - l_arr[bs:be+1].min()
    compact = (c_arr[bs:be+1].max() - c_arr[bs:be+1].min()) / width if width > 0 else 1
    return (width <= BASE_MAX_ATR_WIDTH * avg_atr) and (compact <= BASE_COMPACTNESS_MAX)

def classify_move(net, avg_atr):
    t = LEG_ATR_MIN * avg_atr
    return "up" if net >= t else ("down" if net <= -t else "flat")

def measure_legs(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    if bs < LEG_CANDLES or be + LEG_CANDLES >= len(c_arr): return None
    return (classify_move(c_arr[bs-1] - o_arr[bs-LEG_CANDLES], avg_atr),
            classify_move(c_arr[be+LEG_CANDLES] - c_arr[be],   avg_atr),
            avg_atr)

def proximal_distal(bs, be, zone_type):
    bh, bl = h_arr[bs:be+1], l_arr[bs:be+1]
    return (bh.max(), bl.min()) if zone_type == "demand" else (bl.min(), bh.max())

def check_departure(proximal, be, zone_type, zone_width, avg_atr):
    end = min(len(c_arr)-1, be + DEPARTURE_CANDLES)
    ch  = h_arr[be+1:end+1]; cl = l_arr[be+1:end+1]
    if len(ch) == 0: return 0, 0, 0, False
    dep = (ch.max() - proximal) if zone_type == "demand" else (proximal - cl.min())
    dr  = dep / zone_width if zone_width > 0 else 0
    da  = dep / avg_atr   if avg_atr   > 0 else 0
    return round(dep,3), round(dr,2), round(da,2), (dr >= DEPARTURE_RATIO_MIN and da >= DEPARTURE_ATR_MIN)

def detect_zones():
    results = []
    for bs, be in find_base_clusters():
        if not cluster_ok(bs, be): continue
        legs = measure_legs(bs, be)
        if legs is None: continue
        lin_dir, lout_dir, avg_atr = legs
        pair = FORMATION_MAP.get((lin_dir, lout_dir))
        if pair is None: continue
        formation, zone_type = pair
        prox, dist = proximal_distal(bs, be, zone_type)
        zw = abs(prox - dist)
        dep, dep_ratio, dep_atr, passed = check_departure(prox, be, zone_type, zw, avg_atr)
        if not passed: continue
        results.append(dict(
            bs=bs, be=be, formation=formation, zone_type=zone_type,
            proximal=prox, distal=dist, zone_width=zw,
            departure=dep, dep_ratio=dep_ratio, dep_atr=dep_atr,
        ))
    return results

zones = detect_zones()
print(f"{len(zones)} zones detected")


## 2. `curve_score` — score each zone by its position in the HTF range

In [ ]:
# For this single-timeframe fixture we treat the full dataset range as "HTF"
htf_high = h_arr.max()
htf_low  = l_arr.min()
htf_range = htf_high - htf_low
print(f"HTF range: {htf_low:.2f} → {htf_high:.2f}  (span = {htf_range:.2f})")

def curve_score(zone):
    pos = (zone["proximal"] - htf_low) / htf_range if htf_range > 0 else 0.5
    if pos < 0.333:   third = "low"
    elif pos > 0.667: third = "high"
    else:             third = "mid"
    if zone["zone_type"] == "demand":
        return {"low": 2, "mid": 1, "high": 0}[third], third
    else:
        return {"low": 0, "mid": 1, "high": 2}[third], third

for z in zones:
    z["curve_score"], z["curve_third"] = curve_score(z)
    z["scenario"] = labeled["scenario"].iloc[z["bs"]]

print(f"\n{'Scenario':<22} {'Type':8} {'Third':5} {'Score':>5}")
print("-" * 46)
for z in zones:
    print(f"{z['scenario']:<22} {z['zone_type']:8} {z['curve_third']:5} {z['curve_score']:>5}")


## 3. Visualization — HTF thirds and zone positions

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"
COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"
EDGE       = {"demand": "#26a69a", "supply": "#ef5350"}

third = htf_range / 3
SCORE_COLOR = {0: "#ef5350", 1: "#ffb74d", 2: "#26a69a"}

fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df.index, open=df["open"], high=df["high"],
    low=df["low"], close=df["close"],
    increasing_line_color=COLOR_BULL, decreasing_line_color=COLOR_BEAR, name="Price",
))

# HTF range bands
for level, label, colour, opacity in [
    (htf_low,           "HTF Low",  "#26a69a", 0.04),
    (htf_low + third,   "Mid",      "#7c83fd", 0.04),
    (htf_low + 2*third, "HTF High", "#ef5350", 0.04),
]:
    fig.add_hrect(y0=level, y1=level + third,
                  fillcolor=colour, opacity=opacity, line_width=0)
    fig.add_hline(y=level, line=dict(color=colour, width=0.8, dash="dash"))

for z in zones:
    x0, x1 = df.index[z["bs"]], df.index[z["be"]]
    c_edge  = SCORE_COLOR[z["curve_score"]]
    fig.add_vrect(x0=x0, x1=x1, fillcolor="rgba(255,255,255,0.05)",
                  line=dict(color=c_edge, width=1.5, dash="dot"))
    fig.add_annotation(x=x0, y=z["proximal"],
                       text=f"{z['formation']} curve={z['curve_score']}",
                       showarrow=False, font=dict(size=8, color=c_edge),
                       xanchor="left", yanchor="bottom")

fig.update_layout(
    title="Curve Position Score per Zone",
    height=540, plot_bgcolor=BG, paper_bgcolor=BG, font_color="white",
    xaxis_rangeslider_visible=False, hovermode="x unified",
    xaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False),
    yaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False, title="Price"),
)
fig.show()
